**Importing libraries**

In [5]:
import pandas as pd
import numpy as np
from google.colab import files
from sklearn.metrics import cohen_kappa_score

**Uploading the annotated file**

In [6]:
uploaded_file = files.upload()

file_name = next(iter(uploaded_file))

Saving Two People Annotation.xlsx to Two People Annotation (1).xlsx


**Reading excel file**

In [7]:
df = pd.read_excel(file_name)

print("Dataset successfully uploaded!")
print("Total dataset rows:", len(df))

Dataset successfully uploaded!
Total dataset rows: 700


**Defining annotator columns**

In [8]:
annotator1 = 'Tasnim_Annot'
annotator2 = 'Joy_Annot'

annotator_columns = [annotator1, annotator2]

# Checking whether required columns exist
for col in annotator_columns:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in dataset!")


**Finding missing rows **

In [11]:
# Treating empty strings and whitespace as missing values
df[annotator_columns] = df[annotator_columns].replace(
    r'^\s*$',
    pd.NA,
    regex=True
)

# Finding rows with at least one missing annotation
missing_df = df[
    df[annotator_columns].isna().any(axis=1)
].copy()

print("MISSING ANNOTATION REPORT")

print("Total dataset rows:", len(df))

print("Missing annotation rows:", len(missing_df))

print("\nMissing values by annotator:")
print(df[annotator_columns].isna().sum())

# Keeping only rows with complete annotations
clean_df = df.dropna(
    subset=annotator_columns
).copy()

print("\nValid annotation pairs:", len(clean_df))

print("Excluded rows:", len(missing_df))

MISSING ANNOTATION REPORT
Total dataset rows: 700
Missing annotation rows: 0

Missing values by annotator:
Tasnim_Annot    0
Joy_Annot       0
dtype: int64

Valid annotation pairs: 700
Excluded rows: 0


**Finding agreement and disagreement rows**

In [13]:
'''Agreement: Both annotators assigned the same label
Disagreement: Annotators assigned different labels'''

agreement_df = clean_df[
    clean_df[annotator1] == clean_df[annotator2]
].copy()


disagreement_df = clean_df[
    clean_df[annotator1] != clean_df[annotator2]
].copy()

print("AGREEMENT REPORT")

print("Agreement rows:", len(agreement_df))

print("Disagreement rows:", len(disagreement_df))

if len(clean_df) > 0:
    agreement_percentage = (
        len(agreement_df) / len(clean_df)
    ) * 100

    disagreement_percentage = (
        len(disagreement_df) / len(clean_df)
    ) * 100

    print(f"Percentage agreement: {agreement_percentage:.2f}%")

    print(f"Percentage disagreement: {disagreement_percentage:.2f}%")


AGREEMENT REPORT
Agreement rows: 590
Disagreement rows: 110
Percentage agreement: 84.29%
Percentage disagreement: 15.71%


**Calculating Cohen's Kappa**

In [14]:
if len(clean_df) == 0:
    raise ValueError(
        "No valid annotation pairs available for Kappa calculation."
    )

kappa = cohen_kappa_score(
    clean_df[annotator1],
    clean_df[annotator2]
)

print("COHEN'S KAPPA CALCULATION")

print("Cohen's Kappa:", round(kappa, 4))

COHEN'S KAPPA CALCULATION
Cohen's Kappa: 0.6542


**Calculating weighted Cohen's Kappa**

In [15]:
weighted_kappa = cohen_kappa_score(
    clean_df[annotator1],
    clean_df[annotator2],
    weights='quadratic'
)

print("WEIGHTED COHEN'S KAPPA Calculation")

print("Weighted Cohen's Kappa:", round(weighted_kappa, 4))

WEIGHTED COHEN'S KAPPA Calculation
Weighted Cohen's Kappa: 0.7844


**Exporting output files**

In [16]:
# Agreement rows
agreement_df.to_excel(
    'Agreement_Rows.xlsx',
    index=False
)

# Disagreement rows
disagreement_df.to_excel(
    'Disagreement_Rows.xlsx',
    index=False
)

# Missing annotation rows
missing_df.to_excel(
    'Missing_Annotation_Rows.xlsx',
    index=False
)

# Clean dataset containing only valid annotation pairs
clean_df.to_excel(
    'Clean_Annotation_Dataset.xlsx',
    index=False
)


**Creating a summary report**

In [17]:
summary_df = pd.DataFrame({
    'Metric': [
        'Total Dataset Rows',
        'Valid Annotation Pairs',
        'Missing Annotation Rows',
        'Agreement Rows',
        'Disagreement Rows',
        'Percentage Agreement',
        'Percentage Disagreement',
        "Cohen's Kappa",
        "Weighted Cohen's Kappa"
    ],

    'Value': [
        len(df),
        len(clean_df),
        len(missing_df),
        len(agreement_df),
        len(disagreement_df),
        round(agreement_percentage, 2),
        round(disagreement_percentage, 2),
        round(kappa, 4),
        round(weighted_kappa, 4)
    ]
})

summary_df.to_excel(
    'Kappa_Summary_Report.xlsx',
    index=False
)


**Downloading output files**

In [18]:
print("The following files have been generated:")

output_files = [
    'Agreement_Rows.xlsx',
    'Disagreement_Rows.xlsx',
    'Missing_Annotation_Rows.xlsx',
    'Clean_Annotation_Dataset.xlsx',
    'Kappa_Summary_Report.xlsx'
]

for output_file in output_files:
    print(output_file)
    files.download(output_file)


The following files have been generated:
Agreement_Rows.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Disagreement_Rows.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (5 outputs left out)

<IPython.core.display.Javascript object>

Kappa_Summary_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>